# Semantic Similarity and Plagiarism Detection
**Experiential Learning, Group 4 (Task 4)**

**Goal:** compare two documents, find passages that mean the same thing, and show why *semantic similarity is not the same as plagiarism*.

**Pipeline:** split into sentences, embed each sentence with a pretrained Sentence Transformer, compare with cosine similarity, then check word overlap to separate "copied" from "paraphrased".

| Item | Choice |
|---|---|
| Model | `sentence-transformers/all-MiniLM-L6-v2` (22M parameters, runs on CPU) |
| Dataset 1 | STS-B (`sentence-transformers/stsb`): human similarity scores |
| Dataset 2 | MRPC (`nyu-mll/glue`, config `mrpc`): paraphrase yes/no labels |
| Baseline | TF-IDF + cosine similarity (words only, no meaning) |

## 1. Setup

In [ ]:
!pip install -q sentence-transformers datasets scikit-learn scipy matplotlib pandas

In [ ]:
import re, difflib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import HTML, display
from scipy.stats import spearmanr, pearsonr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
enc = lambda x: model.encode(x, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
pair_scores = lambda a, b: (enc(a) * enc(b)).sum(1)  # cosine similarity of aligned sentence lists

def tfidf_scores(a, b):
    v = TfidfVectorizer().fit([*a, *b])
    return np.asarray(v.transform(a).multiply(v.transform(b)).sum(1)).ravel()

## 2. Core idea: sentences become vectors
Each sentence becomes a 384-number vector. Sentences with similar **meaning** get vectors that point the same way, and **cosine similarity** measures that angle (1.0 = same direction, about 0 = unrelated).

In [ ]:
cat = "The cat sat on the mat."
demo = {"Exact copy": cat, "Paraphrase": "A feline was resting on the rug.",
        "Same topic": "Dogs are loyal animals.", "Unrelated": "The stock market fell sharply today."}
for (name, s), c in zip(demo.items(), pair_scores([cat] * 4, list(demo.values()))):
    print(f"{name:12s} cosine = {c:.3f}   | {s}")

The paraphrase shares almost **no words** with the original, yet scores about 0.56, far above the unrelated sentences (about 0.08). Embeddings find similarity that keyword matching misses, but a paraphrase can still land below a strict threshold, so the threshold needs care.

## 3. Evaluation on STS-B (does the model agree with humans?)
STS-B pairs are rated by humans (stored as 0-1). We correlate the model's cosine scores with those ratings: higher means better agreement.

In [ ]:
stsb = load_dataset("sentence-transformers/stsb", split="test")
a, b, h = stsb["sentence1"], stsb["sentence2"], np.array(stsb["score"])
st_scores = {"Sentence Transformer": pair_scores(a, b), "TF-IDF baseline": tfidf_scores(a, b)}
print("STS-B test pairs:", len(h))
display(pd.DataFrame({k: {"Spearman": spearmanr(s, h)[0], "Pearson": pearsonr(s, h)[0]} for k, s in st_scores.items()}).T.round(3))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
for x, (k, s), c in zip(ax, st_scores.items(), ("#0B7A75", "#C8553D")):
    x.scatter(h, s, s=6, alpha=.35, color=c); x.set(title=f"{k} (Spearman {spearmanr(s, h)[0]:.3f})", xlabel="Human similarity score (0-1)")
ax[0].set_ylabel("Cosine similarity"); plt.tight_layout(); plt.show()

## 4. Evaluation on MRPC (can a threshold detect paraphrases?)
MRPC pairs are labelled **paraphrase (1)** or **not (0)**. For each method we pick the cosine threshold with the best F1 and report **accuracy** and F1, next to a trivial "always paraphrase" reference.

In [ ]:
mrpc = load_dataset("nyu-mll/glue", "mrpc", split="validation")
a, b, y = mrpc["sentence1"], mrpc["sentence2"], np.array(mrpc["label"])
ts, rows, curves = np.arange(0.1, 0.99, 0.01), [], {}
for name, s in {"Sentence Transformer": pair_scores(a, b), "TF-IDF baseline": tfidf_scores(a, b)}.items():
    curves[name] = [f1_score(y, s >= t) for t in ts]
    t = ts[np.argmax(curves[name])]
    rows.append([name, round(t, 2), accuracy_score(y, s >= t), max(curves[name])])
rows.append(["Always says 'paraphrase'", "-", accuracy_score(y, np.ones_like(y)), f1_score(y, np.ones_like(y))])
print(f"MRPC validation pairs: {len(y)}, paraphrases: {y.mean():.1%}")
display(pd.DataFrame(rows, columns=["Method", "Best threshold", "Accuracy", "F1"]).round(3))

plt.figure(figsize=(5.5, 3.6))
for (k, f), c in zip(curves.items(), ("#0B7A75", "#C8553D")): plt.plot(ts, f, label=k, color=c, lw=2)
plt.xlabel("Cosine threshold"); plt.ylabel("F1 score (MRPC)"); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

MRPC is hard and unbalanced (most pairs are paraphrases), so the gain over the baselines is modest. The best threshold is chosen on the same pairs it is reported on, so the figures are slightly optimistic.

## 5. Document comparison with a plagiarism-style label
Meaning alone cannot separate copying from rewording, so we add **word overlap** (same words in the same order):

| Semantic similarity | Word overlap | Label |
|---|---|---|
| high (>= 0.65) | high (>= 0.75) | **Likely copied** |
| high | low | **Paraphrase / same idea** |
| low | any | Not flagged |

Every label is only a **flag for a human reviewer**, never proof of plagiarism.

In [ ]:
SEM_T, LEX_T = 0.65, 0.75
split = lambda t: [s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if len(s.split()) >= 4]
overlap = lambda a, b: difflib.SequenceMatcher(None, a.lower().split(), b.lower().split()).ratio()
label = lambda sem, lex: "Not flagged" if sem < SEM_T else "Likely copied" if lex >= LEX_T else "Paraphrase / same idea"

def compare_documents(doc_a, doc_b):
    sa, sb = split(doc_a), split(doc_b)
    sim = enc(sa) @ enc(sb).T
    df = pd.DataFrame([(a, sb[j], sim[i, j], overlap(a, sb[j])) for i, (a, j) in enumerate(zip(sa, sim.argmax(1)))],
                      columns=["Sentence_A", "Match_in_B", "Semantic", "Lexical"])
    df["Label"] = [label(s, l) for s, l in zip(df.Semantic, df.Lexical)]
    return sim, df

COLORS = {"Likely copied": "#f8b4b4", "Paraphrase / same idea": "#fde68a", "Not flagged": "#e5e7eb"}
show_report = lambda df: display(HTML("".join(
    f"<div style='background:{COLORS[r.Label]};color:#111;padding:8px;margin:6px 0;border-radius:6px;font-family:sans-serif'>"
    f"<b>{r.Label}</b> (semantic {r.Semantic:.2f}, lexical {r.Lexical:.2f})<br><i>A:</i> {r.Sentence_A}<br><i>B:</i> {r.Match_in_B}</div>"
    for r in df.itertuples())))

### 5.1 Sample documents
Text B covers every case: sentence 1 is copied word for word, sentences 2, 3 and 5 are paraphrased, and sentence 4 is unrelated.

In [ ]:
doc_a = '''
Climate change refers to long-term shifts in temperatures and weather patterns.
Burning fossil fuels such as coal, oil and gas is the main driver of global warming.
Renewable energy sources like solar and wind power can reduce greenhouse gas emissions.
Many countries have pledged to reach net-zero emissions by the middle of this century.
Rising sea levels threaten millions of people living in coastal cities.
'''
doc_b = '''
Climate change refers to long-term shifts in temperatures and weather patterns.
The main cause of the rise in global temperatures is the combustion of coal, oil and natural gas.
Wind and solar generation are able to cut emissions of heat-trapping gases.
My favourite sport is cricket because it brings friends and family together.
Coastal cities with millions of residents are at risk as the oceans keep rising.
'''
sim, report = compare_documents(doc_a, doc_b)
report.round(3)

### 5.2 Similarity heatmap (rows: sentences of A, columns: sentences of B)

In [ ]:
plt.figure(figsize=(5.2, 4.2)); plt.imshow(sim, vmin=0, vmax=1, cmap="viridis"); plt.colorbar(label="Cosine similarity")
plt.xticks(range(sim.shape[1]), [f"B{i+1}" for i in range(sim.shape[1])]); plt.yticks(range(sim.shape[0]), [f"A{i+1}" for i in range(sim.shape[0])])
for (i, j), v in np.ndenumerate(sim): plt.text(j, i, f"{v:.2f}", ha="center", va="center", color="white", fontsize=8)
plt.title("Sentence-to-sentence similarity"); plt.tight_layout(); plt.show()

### 5.3 Highlighted report

In [ ]:
show_report(report)

### 5.4 Try your own text
Edit the two strings and re-run the cell.

In [ ]:
my_a = "Machine learning lets computers learn patterns from data without being explicitly programmed."
my_b = "Computers can discover patterns in data by themselves, so no explicit programming is required."
show_report(compare_documents(my_a, my_b)[1])

### 5.5 Accuracy of the full labeller
The benchmarks above test the embeddings. Here we test the **three-way label** itself on 18 sentence pairs we wrote and labelled by hand (6 per class). The thresholds stay fixed at 0.65 and 0.75. This is a small, hand-made set, so treat it as a sanity check rather than a benchmark.

In [ ]:
pairs = {
    "Likely copied": [
        ("The Earth revolves around the Sun once every year.", "The Earth revolves around the Sun once every year."),
        ("Photosynthesis allows plants to convert sunlight into chemical energy.", "Photosynthesis allows plants to convert sunlight into chemical energy stored in sugar."),
        ("The Industrial Revolution began in Britain in the eighteenth century.", "The Industrial Revolution began in Britain in the 18th century."),
        ("Regular exercise reduces the risk of heart disease and improves mental health.", "Regular exercise reduces the risk of heart disease and improves mental well-being."),
        ("The Indian Constitution came into effect on 26 January 1950.", "The Indian Constitution came into force on 26 January 1950."),
        ("Machine learning models learn patterns from large amounts of data.", "Machine learning models learn patterns from very large amounts of data.")],
    "Paraphrase / same idea": [
        ("The government announced a new scheme to support small farmers.", "A fresh programme to help farmers with small landholdings was unveiled by the authorities."),
        ("Heavy rain caused flooding in several parts of the city.", "Several areas of the city were inundated after intense rainfall."),
        ("Students who sleep well tend to score higher in exams.", "Good sleep is linked to better exam results among learners."),
        ("The company's profits fell sharply because of rising costs.", "Increased expenses led to a steep drop in the firm's earnings."),
        ("She was too tired to continue working on the project.", "Exhaustion kept her from carrying on with the assignment."),
        ("Electric vehicles produce no exhaust emissions while driving.", "Cars powered by batteries do not release tailpipe pollution on the road.")],
    "Not flagged": [
        ("The Earth revolves around the Sun once every year.", "My brother bought a new laptop yesterday."),
        ("Heavy rain caused flooding in several parts of the city.", "The football match ended in a draw."),
        ("Regular exercise reduces the risk of heart disease.", "The museum opens at nine in the morning."),
        ("The government announced a new scheme to support small farmers.", "The government announced a new date for the annual sports meet."),
        ("Photosynthesis allows plants to convert sunlight into chemical energy.", "Plants need water and soil to grow in gardens."),
        ("Electric vehicles produce no exhaust emissions while driving.", "Petrol prices rose again this week across the country.")]}
test = pd.DataFrame([(a, b, t) for t, ps in pairs.items() for a, b in ps], columns=["A", "B", "Truth"])
test["Pred"] = [label(s, overlap(a, b)) for s, a, b in zip(pair_scores(list(test.A), list(test.B)), test.A, test.B)]
print(f"Labeller accuracy: {(test.Truth == test.Pred).mean():.1%} ({(test.Truth == test.Pred).sum()} of {len(test)} correct)")
pd.crosstab(test.Truth, test.Pred)

## 6. Semantic similarity is not plagiarism
Two sentences can score very high **without any copying**:

In [ ]:
fact = ["Water boils at 100 degrees Celsius at sea level.", "At sea level, water reaches its boiling point at 100 degrees Celsius."]
print("Cosine similarity:", round(float(pair_scores(fact[:1], fact[1:])[0]), 3))

Two students can independently state the same fact and get a high score. A flag only means *"look at this passage"*. A real plagiarism decision also needs:
- **Attribution and intent:** was the source cited? Is it a quotation?
- **Common knowledge:** definitions, standard facts and stock phrases always match.
- **Context:** the same idea may be legitimately restated.
- **Human review** of every flagged passage.

**Limitations**
- Thresholds depend on the domain and must be tuned.
- `all-MiniLM-L6-v2` is built for English, and long sentences are truncated at 256 word-pieces.
- It compares two documents you provide; it does not search the internet or a source database.
- Sentence-level matching misses plagiarism spread across several sentences.

## 7. Summary
| Part | What it shows |
|---|---|
| STS-B | Embeddings agree with human similarity ratings better than TF-IDF (section 3) |
| MRPC | Accuracy and F1 of a cosine threshold for paraphrase detection (section 4) |
| Labeller | Accuracy of the three-way label on 18 hand-labelled pairs (section 5.5) |
| Key point | Semantic similarity is evidence for human review, not proof of plagiarism |